In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Leer desde la capa de bronce

In [0]:
from pyspark.sql.functions import col, when,trim,create_map, regexp_replace,lit,to_date,rlike, expr,coalesce, lower,broadcast, max, min, avg, stddev,row_number, sum as suma
from itertools import chain
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from clean import normalizar_fecha_spark,normalizar_nombre_columna
from dict_data import map_tipo_vac, dict_transformacion_saps


In [0]:
df_inmu=spark.table("saps_ctes.bronze.datosctes_inmunizaciones")
df_saps=spark.table("saps_ctes.silver.datosctes_saps")
df_inmu.limit(10).display()

# Realizar las transformaciones y limpieza

## Análisis inicial de las columnas

Revisemos primeros los valores para cada columna

In [0]:
df_inmu.columns

De las columnas presentes, sólo nos vamos a quedar con las siguientes:
ID_SAPS',
'SAPS',
'fecha',
'vacunas_tipo',
'vacunas_cantidad',

In [0]:
df_inmu=df_inmu.select('ID_SAPS','SAPS','fecha','vacunas_tipo','vacunas_cantidad')
df_inmu.columns

Renombremos las columnas 'ID_SAPS' y 'SAPS' a minúscula.

In [0]:
df_inmu=df_inmu.withColumnRenamed('ID_SAPS','id_saps').withColumnRenamed('saps','saps')

df_inmu.columns


Una vez renombradas las columnas, vamos a realizar los análisis pertinentes a cada columna. Primero, busquemos si existen nulos.

In [0]:
#campos con id nulos
df_inmu.filter(col('id_saps').isNull()).count()

In [0]:
#id_saps nulos, pero saps no nulos. Podemos usar el nombre del saps para rellenar
#los id
df_inmu.filter((col('id_saps').isNull()) & (col('saps').isNotNull())).count()

In [0]:
#id_saps y saps nulos. Podemos completar estos elementos con -1 y n/a
df_inmu.filter((col('id_saps').isNull()) & (col('saps').isNull())).count()

In [0]:
#registros con la cantidad de vacunas en nulo
df_inmu.filter(col('vacunas_cantidad').isNull()).count()

In [0]:
#registros con el tipo de vacuna en nulo
df_inmu.filter(col('vacunas_tipo').isNull()).count()

In [0]:
#ver los valores que no sean sólo números en el campo vacunas cantidad
df_inmu.filter(~col('vacunas_cantidad').rlike("^[0-9]+$")).select('vacunas_cantidad').distinct().display()


In [0]:
#ver los valores que no sean sólo números en el campo id_saps
df_inmu.filter(~col('id_saps').rlike("^[0-9]+$")).select('id_saps').distinct().display()

# Trabajando Con Las Fechas

Primero, vamos a normalizar las fechas para que sean del tipo yyyy-mm-dd

In [0]:
df_inmu = df_inmu.withColumn(
    "fecha",
    regexp_replace(col("fecha"), "/", "-")
)
df_inmu = df_inmu.withColumn(
    "fecha",
    normalizar_fecha_spark(col("fecha"))
)

In [0]:
df_inmu.select('fecha').distinct().display()

# Trabajando Con Los Tipos De Vacunas

In [0]:
df_inmu = df_inmu.withColumn(
    "vacunas_tipo",
    trim(regexp_replace(col("vacunas_tipo"), r"\s+", " "))
)
df_inmu.select('vacunas_tipo').distinct().orderBy('vacunas_tipo').display()

In [0]:



#realizo el mapeo
expr = create_map([lit(x) for kv in map_tipo_vac.items() for x in kv])

#luego aplico el mapeo
df_inmu = df_inmu.withColumn(
    "vacunas_tipo",
    coalesce(expr.getItem(col("vacunas_tipo")),col("vacunas_tipo"))
)

Lo que se hizo arriba fue homologar los nombres. Por ejemplo, había vacunas como "NEUMO 13" y "NEUMO13" que hacían referencia a la misma vacuna. Con esto, igualamos los nombres para no tratarlas como vacunas diferentes.

# Trabajando Con Las Cantidades De Vacunas

Ya hemos encontrado dos valores peculiares: "-" y "´1". Reemplazaremos estos valores por None.

In [0]:
#este código controla si los valores del campo vacunas_cantidad son todos números
#si lo son, conserva el valor; si no lo son, los transforma en nulo
df_inmu = df_inmu.withColumn(
    "vacunas_cantidad",
    when(
        col("vacunas_cantidad").rlike("^[0-9]+$"),
        col("vacunas_cantidad")
    ).otherwise(None))

# Trabajando con los id_saps y saps

Lo que debemos controlar son los valores de los id_saps para saber si son los mismos que en el listado de saps y lo mismo para los nombres de los saps.

In [0]:
df_inmu.select('id_saps','saps').distinct().display()

Notamos que en id_saps hay valores nulos, pero también están los valores '#REF!' y '#N/A'. Cambiemos esos valores a None.

In [0]:
#elimino espacios en blanco al principio y final
df_inmu = df_inmu.withColumn(
    "saps",
    trim(
        regexp_replace(col("saps"), r"[\s\u00A0]+", " ")
    )
)
#tranformo los campos que no sean sólo enteros en None
df_inmu = df_inmu.withColumn(
    "id_saps",
    when(
        col("id_saps").rlike("^[0-9]+$"),
        col("id_saps")
    ).otherwise(None))

Ahora, adaptemos los nombres de los saps para las inmunizaciones de tal manera que sean exactamente igual que los nombres que figuran en el listado de saps

In [0]:
mapping_expr_saps = create_map([lit(x) for x in chain(*dict_transformacion_saps.items())])
df_inmu = df_inmu.withColumn(
    "saps",
    coalesce(mapping_expr_saps.getItem(col("saps")),col("saps"))
)

Una vez estandarizado los nombres, lo que debemos hacer es aquellos registros que tengan el nombre del saps pero no su id, darle el id dependiendo del nombre del saps.

In [0]:
#primero, recupero la información del saps y del id para los saps
df_saps_id=df_saps.select(col('id_saps').alias('id_saps_map'),col('saps')).distinct()
df_saps_id.display()

In [0]:
df_inmu=df_inmu.join(broadcast(df_saps_id),on='saps',how="left").withColumn('id_saps',coalesce(col('id_saps'),col('id_saps_map'))).drop('id_saps_map')
    


In [0]:
df_inmu.select('id_saps','saps').distinct().display()

De los resultados notamos que el centro de distribución de vacunas no posee ningún id. Así que vamos asignarle uno arbitrario: le daremos un id que sea igual al máximo id presente en id_saps.

In [0]:
df_inmu=df_inmu.withColumn('id_saps',col('id_saps').cast('int'))

In [0]:
max_id=df_inmu.select(max('id_saps')).collect()[0][0]
print(max_id)



In [0]:
df_inmu=(df_inmu.withColumn(
            'id_saps',
            when(col('saps')=='Centro de distribucion de Vacunas',lit(max_id+1))
            .otherwise(col('id_saps'))
            )
        )
        

# Completando los valores nulos de los campos

Seguiremos el mismo criterio: los campos string se completan con 'n/a', los campos numéricos con -1 y las fechas con 1900-01-01

In [0]:
df_inmu = (
    df_inmu.withColumn('id_saps', coalesce(col('id_saps'), lit(-1)))
    .withColumn('saps', coalesce(col('saps'), lit('n/a')))
    .withColumn('vacunas_cantidad', coalesce(col('vacunas_cantidad'), lit(-1)))
    .withColumn('vacunas_tipo', coalesce(col('vacunas_tipo'), lit('n/a')))
)

# Elimnando los duplicados

In [0]:
df_inmu=df_inmu.dropDuplicates()

# Pasar a la capa de plata

In [0]:
%sql
DROP TABLE IF EXISTS saps_ctes.silver.datosctes_inmunizaciones;

In [0]:
df_inmu.write.mode("overwrite").saveAsTable("saps_ctes.silver.datosctes_inmunizaciones")